# 02. Exploratory Data Analysis & Standards Verification

**Course:** URP4201 Research Methodology

Verifying physicochemical parameters against World Health Organization (WHO) and Bureau of Indian Standards (BIS IS 10500:2012) thresholds.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "AquaSense AI" else Path(".").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.aquasense.data import FEATURE_COLUMNS, TARGET_COLUMN, load_raw_dataset
from src.aquasense.features import WHO_BIS_STANDARDS

df = load_raw_dataset("data/raw/water_quality.csv")
print("Dataset successfully loaded for EDA. Total shape:", df.shape)

Dataset successfully loaded for EDA. Total shape: (1000, 10)


In [2]:
# Statutory WHO / BIS threshold breach rates
breach_summary = []
for col, limits in WHO_BIS_STANDARDS.items():
    if col in df.columns:
        min_limit = limits.get("min", -np.inf)
        max_limit = limits.get("max", limits.get("clean_max", np.inf))
        breaches = ((df[col] < min_limit) | (df[col] > max_limit)).sum()
        breach_summary.append({
            "Parameter": col,
            "Statutory Min": min_limit if min_limit != -np.inf else "-",
            "Statutory Max": max_limit if max_limit != np.inf else "-",
            "Breach Count": breaches,
            "Breach Rate (%)": round((breaches / len(df)) * 100, 2)
        })
pd.DataFrame(breach_summary)

,Parameter,Statutory Min,Statutory Max,Breach Count,Breach Rate (%)
0,pH,6.5,8.5,330,33.0
1,Turbidity,-,5.0,391,39.1
2,Dissolved Oxygen,6.5,-,660,66.0
3,Biochemical Oxygen Demand,-,2.0,541,54.1
4,Lead,-,0.01,376,37.6
5,Mercury,-,0.001,347,34.7
6,Arsenic,-,0.01,375,37.5


In [3]:
# Pairwise Pearson correlation matrix between water quality features
corr = df[FEATURE_COLUMNS].corr()
corr.round(3)

,pH,Turbidity,Temperature,Dissolved Oxygen,Biochemical Oxygen Demand,Lead,Mercury,Arsenic
pH,1.000,-0.051,0.014,0.057,0.010,-0.013,-0.017,-0.100
Turbidity,-0.051,1.000,0.024,-0.651,0.645,0.527,0.608,0.602
Temperature,0.014,0.024,1.000,-0.267,0.023,-0.002,0.004,0.006
Dissolved Oxygen,0.057,-0.651,-0.267,1.000,-0.640,-0.587,-0.589,-0.645
Biochemical Oxygen Demand,0.010,0.645,0.023,-0.640,1.000,0.586,0.545,0.584
Lead,-0.013,0.527,-0.002,-0.587,0.586,1.000,0.498,0.544
Mercury,-0.017,0.608,0.004,-0.589,0.545,0.498,1.000,0.510
Arsenic,-0.100,0.602,0.006,-0.645,0.584,0.544,0.510,1.000


In [4]:
# Mean parameter levels stratified by risk category
df.groupby(TARGET_COLUMN)[FEATURE_COLUMNS].mean().rename(index={0: "Good", 1: "Moderate", 2: "Polluted"}).T.round(4)

Pollution Level,Good,Moderate,Polluted
pH,7.2750,6.5902,7.1093
Turbidity,1.5003,6.4426,24.9030
Temperature,22.1702,22.0897,22.2634
Dissolved Oxygen,6.8677,4.8685,2.5884
Biochemical Oxygen Demand,1.2830,3.9348,13.3317
Lead,0.0030,0.0109,0.0499
Mercury,0.0002,0.0010,0.0046
Arsenic,0.0028,0.0109,0.0389
